<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c02-header.png" alt="Nextia Learning · Python for Practical AI Engineering" width="100%">

# Structured records

**[Structured records](https://learning.nextia-ai.com/courses/python/m03/structured-records/)** · Python for Practical AI Engineering · Module 3, lesson 3 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** add type hints that show each function's contract, give each ticket its own type with a dataclass, and separate validation, parsing and reporting into their own modules.

| | |
|---|---|
| **Time** | About 45 minutes |
| **Needs** | Nothing to install, and no internet connection: only Python. |
| **You should know** | The package `ticket_cleaner` with `report.py`, and imports, from [Modules and imports](https://learning.nextia-ai.com/courses/python/m03/modules-and-imports/). |
| **You do not need** | Your local project. The setup makes the same `ticket-cleaner` folder here. |
| **Tested** | Python 3.14.6, in Jupyter on macOS, from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every example, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/python/m03/structured-records/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C02/M03-L03-structured-records/structured-records-solution.ipynb).

## Setup: make the project folder

Run the next cell. It makes a folder `ticket-cleaner`, laid out like your project from Module 1, and works in it. It writes the files that you made in the earlier lessons (the code is in the cell). It needs no internet connection.

You should see `Ready: ticket-cleaner, 4 files`. If you run the cell again, it puts every file back as at the start of this lesson: use this to start again.

The files are the code of these lessons, as you have it at the end of each one: [Decisions and repetition](https://learning.nextia-ai.com/courses/python/m02/decisions-and-repetition/) (count_tickets.py), [Function contracts](https://learning.nextia-ai.com/courses/python/m03/function-contracts/) (count_tickets.py as functions), [Modules and imports](https://learning.nextia-ai.com/courses/python/m03/modules-and-imports/) (the package `ticket_cleaner` with `report.py`).

In [ ]:
# Setup: make the ticket-cleaner folder as at the start of this lesson.
import hashlib, os, shutil, subprocess, sys, urllib.request
from pathlib import Path

if Path.cwd().name == "ticket-cleaner":  # the cell ran before: start from the parent folder
    os.chdir("..")
PROJECT = Path("ticket-cleaner").resolve()
for item in PROJECT.glob("*"):  # start again: remove everything except the data files
    if item.name != "data":
        shutil.rmtree(item) if item.is_dir() else item.unlink()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)


FILES = {
    'count_tickets.py': '''\
from ticket_cleaner.report import average_priority, count_by_category, filter_by_status

TICKETS = [
    {"id": "T-1001", "status": "open", "category": "billing", "priority": 2},
    {"id": "T-1002", "status": "closed", "category": "login", "priority": 3},
    {"id": "T-1003", "status": "open", "category": "login", "priority": 1},
    {"id": "T-1004", "status": "pending", "category": "billing", "priority": 2},
    {"id": "T-1005", "status": "open", "category": "billing", "priority": 3},
    {"id": "T-1006", "status": "closed", "category": "shipping", "priority": 1},
]


def main():
    status = "open"
    selected = filter_by_status(TICKETS, status)
    print(f"Tickets with status '{status}': {len(selected)}")
    if not selected:
        print("No tickets to count.")
        return
    for category, count in count_by_category(selected).items():
        print(f"  {category}: {count}")
    print(f"Average priority: {average_priority(selected)}")


if __name__ == "__main__":
    main()
''',
    'requirements.txt': '''\
httpx==0.28.1
''',
    'ticket_cleaner/__init__.py': '''\
"""Clean support-ticket records and summarise them."""
''',
    'ticket_cleaner/report.py': '''\
def filter_by_status(tickets, status="open"):
    """Return the tickets that have the given status."""
    selected = []
    for ticket in tickets:
        if ticket["status"] == status:
            selected.append(ticket)
    return selected


def count_by_category(tickets):
    """Return the number of tickets in each category, sorted by category."""
    counts = {}
    for ticket in tickets:
        category = ticket["category"]
        counts[category] = counts.get(category, 0) + 1
    return dict(sorted(counts.items()))


def average_priority(tickets):
    """Return the average priority, or None when there are no tickets."""
    if not tickets:
        return None
    total = 0
    for ticket in tickets:
        total = total + ticket["priority"]
    return round(total / len(tickets), 1)
''',
}
for name, text in FILES.items():
    (PROJECT / name).parent.mkdir(parents=True, exist_ok=True)
    (PROJECT / name).write_text(text, encoding="utf-8")

os.chdir(PROJECT)  # work in the project folder, as in the lesson
sys.path.insert(0, str(PROJECT))
sys.dont_write_bytecode = True  # always read the newest version of each file
count = sum(1 for path in PROJECT.rglob("*") if path.is_file())
print("Ready: ticket-cleaner" + (f", {count} file{'s' if count > 1 else ''}" if count else ""))

## Setup: the helpers

The next cell defines small functions that this notebook uses. You do not need to read them, but you can.

- `check("""assert ...""")` runs the `assert` lines of a check cell. It prints `Check passed.`, or `Not yet:` and what is different. It does not show the answer.
- `run("count_tickets.py")` runs a file of the project with Python, as `python count_tickets.py` does in your terminal, and shows its output and its exit code. `run("-m", "pytest")` is `python -m pytest`. `env={"NAME": "value"}` sets an environment variable for that one run.
- `fresh("ticket_cleaner.parsing")` imports a module of the project again, so that the notebook uses the newest version of its file. (A plain `import` uses the copy that is already in memory: see [Notebook discipline](https://learning.nextia-ai.com/courses/python/m06/notebook-discipline/).)

Run the cell. It prints nothing.

In [ ]:
# Helpers: run the project, and check an answer without showing it.
import os, sys

def check(asserts):
    """Run the assert lines of a check. Say what is different, never the answer."""
    try:
        exec(asserts, globals())
    except AssertionError as error:
        print(f"Not yet: {error}")
    except NameError as error:
        print(f"Not yet: {error}. Run your cell above first.")
    except Exception as error:
        print(f"Not yet: the check stopped with {type(error).__name__}: {error}")
    else:
        print("Check passed.")

import subprocess, sys

def environment(env=None):
    """The environment variables of a run: the notebook's, plus `env` (None
    removes one). No colours, so the output reads as plain text."""
    variables = {**os.environ, "PYTHONUNBUFFERED": "1", "PYTHONDONTWRITEBYTECODE": "1",
                 "PYTHON_COLORS": "0", "NO_COLOR": "1", "PY_COLORS": "0"}
    variables.pop("FORCE_COLOR", None)
    for name, value in (env or {}).items():
        if value is None:
            variables.pop(name, None)
        else:
            variables[name] = value
    return variables

def run_quiet(*args, env=None):
    """Run `python <args>` in the project folder. Return the result
    (.returncode, .stdout, .stderr)."""
    return subprocess.run([sys.executable, *args], env=environment(env), text=True, capture_output=True)

def run(*args, env=None, streams="both"):
    """Run `python <args>` in the project folder, like the terminal, and show
    the output and the exit code. streams="stdout" or "stderr" shows one only."""
    if streams == "both":  # the two streams together, in the order they were written
        result = subprocess.run([sys.executable, *args], env=environment(env), text=True,
                                stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
        output = result.stdout
    else:
        result = run_quiet(*args, env=env)
        output = result.stdout if streams == "stdout" else result.stderr
    if output:
        print(output, end="" if output.endswith("\n") else "\n")
    print(f"(exit code {result.returncode})")

import importlib

def fresh(module_name):
    """Import a module of the project again, from its newest file."""
    for name in list(sys.modules):
        if name == "ticket_cleaner" or name.startswith("ticket_cleaner."):
            del sys.modules[name]
    importlib.invalidate_caches()
    return importlib.import_module(module_name)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. Raw records are not tickets yet

Until now, the tickets were perfect. Real data is not like that. When you read a file in Module 4, each record arrives as a dictionary of strings, typed by people. The first record below is valid after cleaning. The second has a typing mistake in the status. The third has a priority that is not a number.

> **Predict.** What does the filter from `report.py` find, and what does the average do? Then run the cell.

In [ ]:
from ticket_cleaner.report import average_priority, filter_by_status

raw_rows = [
    {"id": "t-1003", "status": "Open ", "category": "login", "priority": "1"},
    {"id": "T-1004", "status": "opne", "category": "billing", "priority": "2"},
    {"id": "T-1005", "status": "open", "category": "billing", "priority": "high"},
]
print(len(filter_by_status(raw_rows)), "open")
try:
    print(average_priority(raw_rows))
except TypeError as error:
    print("TypeError:", error)

> **Check.** You should see `1 open` (it misses `"Open "`), then `TypeError: unsupported operand type(s) for +: 'int' and 'str'`. The report functions cannot trust this input.

So the program gets three steps, each in its own module: **validate** (find the problems in one raw record, in `parsing.py`), **parse** (turn a valid raw record into a clean `Ticket`, in `parsing.py`), and **report** (calculate with `Ticket` objects only, in `report.py`).

## 2. Type hints: the contract on the def line

A **type hint** says the type that a parameter expects, after a `:`, or that a function returns, after `->`. `list[str]` is a list of strings, and `float | None` is a `float` or `None`. Python does **not** check type hints when the program runs. They are for people and for tools such as VS Code.

> **Predict.** The hint says `list[int]`. What happens when you give a list of floats? Then run the cell.

In [ ]:
def average(values: list[int]) -> float:
    return sum(values) / len(values)

print(average([1.5, 2.5]))
print(average.__annotations__)

> **Check.** You should see `2.0`: Python ran the call with no error. The second line shows the hints that the editor reads.

## 3. Dataclasses: a type for a ticket

A **dataclass** is the shortest way to write a class, a new type, that mainly holds data. Each field has a type hint. `frozen=True` means that a ticket cannot change after you make it. `is_urgent` is a **method**: a function that belongs to the type, and `self` is the ticket that you call it on.

Run the cell. The last part has a typing mistake in a field name.

In [ ]:
from dataclasses import dataclass, replace


@dataclass(frozen=True)
class Ticket:
    """One valid support ticket."""

    id: str
    status: str
    category: str
    priority: int

    def is_urgent(self) -> bool:
        """Return True for priority 1, the most urgent level."""
        return self.priority == 1


ticket = Ticket(id="T-1003", status="open", category="login", priority=1)
print(ticket)
print(ticket.status, ticket.is_urgent())

same = Ticket(id="T-1003", status="open", category="login", priority=1)
print("Equal:", ticket == same)

try:
    print(ticket.stauts)
except AttributeError as error:
    print("AttributeError:", error)

> **Check.** You should see `Ticket(id='T-1003', status='open', category='login', priority=1)`, `open True`, `Equal: True`, then `AttributeError: 'Ticket' object has no attribute 'stauts'`. A traceback also suggests the closest name: `Did you mean: 'status'?`

> **Predict.** What happens if you try to change the status of a frozen ticket? Run the cell. Then see how `replace` makes a new ticket with one field changed.

In [ ]:
try:
    ticket.status = "closed"
except AttributeError as error:
    print(type(error).__name__, error)

closed = replace(ticket, status="closed")
print(closed)
print(ticket.status)

> **Check.** You should see `FrozenInstanceError cannot assign to field 'status'`, then the new ticket with `status='closed'`, and `open`: the original did not change. A frozen ticket is safe to share between lists and functions.

## 4. Worked example: validate, then parse

The validation function returns a **list of problems**. An empty list means that the record is valid. Then the report can say **why** a record was rejected, and it finds all the problems in a record at once.

Run the next three cells. They make `records.py` (the `Ticket` type and the allowed values), `parsing.py` (`clean_text`, `find_problems` and `to_ticket`), and the typed `report.py`. Read the lesson page for an explanation of each line.

In [ ]:
%%writefile ticket_cleaner/records.py
from dataclasses import dataclass

STATUSES = {"open", "pending", "closed"}
PRIORITIES = {1, 2, 3}
REQUIRED_FIELDS = ("id", "status", "category", "priority")


@dataclass(frozen=True)
class Ticket:
    """One valid support ticket."""

    id: str
    status: str
    category: str
    priority: int

    def is_urgent(self) -> bool:
        """Return True for priority 1, the most urgent level."""
        return self.priority == 1

In [ ]:
%%writefile ticket_cleaner/parsing.py
from ticket_cleaner.records import PRIORITIES, REQUIRED_FIELDS, STATUSES, Ticket


def clean_text(value: object) -> str:
    """Return the value as text with no outer spaces, in small letters."""
    if value is None:
        return ""
    return str(value).strip().lower()


def find_problems(raw: dict) -> list[str]:
    """Return a list of problems with one raw record. An empty list means valid."""
    problems = []
    for field in REQUIRED_FIELDS:
        if clean_text(raw.get(field)) == "":
            problems.append(f"missing {field}")

    status = clean_text(raw.get("status"))
    if status and status not in STATUSES:
        problems.append(f"unknown status {status!r}")

    priority = clean_text(raw.get("priority"))
    if priority and not (priority.isdigit() and int(priority) in PRIORITIES):
        problems.append(f"priority must be 1, 2 or 3, not {priority!r}")
    return problems


def to_ticket(raw: dict) -> Ticket:
    """Make a Ticket from a raw record that has no problems."""
    return Ticket(
        id=clean_text(raw["id"]).upper(),
        status=clean_text(raw["status"]),
        category=clean_text(raw["category"]),
        priority=int(clean_text(raw["priority"])),
    )

In [ ]:
%%writefile ticket_cleaner/report.py
from ticket_cleaner.records import Ticket


def filter_by_status(tickets: list[Ticket], status: str = "open") -> list[Ticket]:
    """Return the tickets that have the given status."""
    return [ticket for ticket in tickets if ticket.status == status]


def count_by_category(tickets: list[Ticket]) -> dict[str, int]:
    """Return the number of tickets in each category, sorted by category."""
    counts = {}
    for ticket in tickets:
        counts[ticket.category] = counts.get(ticket.category, 0) + 1
    return dict(sorted(counts.items()))


def average_priority(tickets: list[Ticket]) -> float | None:
    """Return the average priority, or None when there are no tickets."""
    if not tickets:
        return None
    return round(sum(ticket.priority for ticket in tickets) / len(tickets), 1)

Try the two parsing functions on the three raw records from section 1. `fresh()` imports the module again from its new file. `{status!r}` in a message puts quotes around the value, so that a value with a space is visible.

> **Predict.** Which problems does each record have? Then run the cell.

In [ ]:
parsing = fresh("ticket_cleaner.parsing")

for raw in raw_rows:
    problems = parsing.find_problems(raw)
    if problems:
        print(raw["id"], problems)
    else:
        print(raw["id"], "->", parsing.to_ticket(raw))

> **Check.** You should see `t-1003 -> Ticket(id='T-1003', status='open', category='login', priority=1)`, `T-1004 ["unknown status 'opne'"]` and `T-1005 ["priority must be 1, 2 or 3, not 'high'"]`. `to_ticket` cleaned the first record. It does not check anything: the caller must call `find_problems` first.

## 5. Guided practice: the script with raw records

`count_tickets.py` now gets raw strings, as from a file. It names each record with a problem, and reports only on the clean tickets. Run the cell, then the `run` cell.

In [ ]:
%%writefile count_tickets.py
from ticket_cleaner.parsing import find_problems, to_ticket
from ticket_cleaner.records import Ticket
from ticket_cleaner.report import average_priority, count_by_category, filter_by_status

SAMPLE_ROWS = [
    {"id": "T-1001", "status": "open", "category": "billing", "priority": "2"},
    {"id": "T-1002", "status": "closed", "category": "login", "priority": "3"},
    {"id": "t-1003", "status": "Open ", "category": "login", "priority": "1"},
    {"id": "T-1004", "status": "opne", "category": "billing", "priority": "2"},
    {"id": "T-1005", "status": "open", "category": "billing", "priority": "high"},
    {"id": "T-1006", "status": "closed", "category": "shipping", "priority": "1"},
]


def main() -> None:
    tickets: list[Ticket] = []
    for raw in SAMPLE_ROWS:
        problems = find_problems(raw)
        if problems:
            print(f"Skipped {raw['id']}: {'; '.join(problems)}")
        else:
            tickets.append(to_ticket(raw))

    status = "open"
    selected = filter_by_status(tickets, status)
    print(f"Tickets with status '{status}': {len(selected)}")
    if not selected:
        print("No tickets to count.")
        return
    for category, count in count_by_category(selected).items():
        print(f"  {category}: {count}")
    print(f"Average priority: {average_priority(selected)}")


if __name__ == "__main__":
    main()

In [ ]:
run("count_tickets.py")

> **Check.** You should see `Skipped T-1004: unknown status 'opne'`, `Skipped T-1005: priority must be 1, 2 or 3, not 'high'`, `Tickets with status 'open': 2`, `  billing: 1`, `  login: 1` and `Average priority: 1.5`. `t-1003` with `"Open "` counts as open, because `to_ticket` cleaned it. The two records with problems are named, not lost.

## 6. Your turn: one more rule

The help desk says that every ticket ID must start with `T-`. This rule is for practice: the cell has its own copy of `find_problems`, and your project does not change, because the next modules expect the rules above.

> **Your turn.** Add the rule to `find_problems` in the cell, at `# Your rule here.` For an ID that does not start with `T-` (after cleaning), add the problem `"id must start with 'T-'"`. A missing ID must give only `"missing id"`, not both problems. Run the cell, then the check cell.

`{**good, "id": "X-9"}` makes a copy of `good` with a different ID.

In [ ]:
STATUSES = {"open", "pending", "closed"}
PRIORITIES = {1, 2, 3}
REQUIRED_FIELDS = ("id", "status", "category", "priority")


def clean_text(value: object) -> str:
    if value is None:
        return ""
    return str(value).strip().lower()


def find_problems(raw: dict) -> list[str]:
    problems = []
    for field in REQUIRED_FIELDS:
        if clean_text(raw.get(field)) == "":
            problems.append(f"missing {field}")

    status = clean_text(raw.get("status"))
    if status and status not in STATUSES:
        problems.append(f"unknown status {status!r}")

    priority = clean_text(raw.get("priority"))
    if priority and not (priority.isdigit() and int(priority) in PRIORITIES):
        problems.append(f"priority must be 1, 2 or 3, not {priority!r}")

    # Your rule here.

    return problems


good = {"id": "t-1003", "status": "open", "category": "login", "priority": "1"}
print(find_problems(good))
print(find_problems({**good, "id": "X-9"}))
print(find_problems({**good, "id": ""}))

In [ ]:
check("""
good = {"id": "t-1003", "status": "open", "category": "login", "priority": "1"}
assert find_problems(good) == [], "A valid record must have no problems. Did you clean the ID before the check?"
assert find_problems({**good, "id": "X-9"}) == ["id must start with 'T-'"], "An ID such as X-9 must give exactly one problem: id must start with 'T-'"
assert find_problems({**good, "id": ""}) == ["missing id"], "A missing ID must give only 'missing id'."
""")

## 7. Failure case: dictionary access on a Ticket

After you change the dictionaries to `Ticket` objects, some code still uses `ticket["status"]`.

> **Predict.** What happens? Then run the cell.

In [ ]:
try:
    print(ticket["status"])
except TypeError as error:
    print("TypeError:", error)

> **Check.** You should see `TypeError: 'Ticket' object is not subscriptable`. A dataclass has fields, not keys. "Subscriptable" means "can be read with `[ ]`".

**Fix:** read the line number in the error, and change `ticket["status"]` to `ticket.status`. In VS Code, look for underlined names before you run the code: the editor knows the fields of `Ticket` from its type hints.

## Check your understanding and recap

Answer the **knowledge checks** and the **module check** on the [lesson page](https://learning.nextia-ai.com/courses/python/m03/structured-records/). They count toward your certificate when you are signed in and enrolled. On your computer, build the three modules in your own project, as in the guided practice of the lesson.

In this lesson you learned that type hints show a function's contract on its `def` line, that a dataclass defines a type for a record, and that the program validates first, parses only valid records, and reports on clean `Ticket` objects only.

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Reading and writing files](https://learning.nextia-ai.com/courses/python/m04/reading-and-writing-files/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/python/m03/structured-records/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The tickets in this course are made up; they have no real people.